In [16]:
# Local Spark session (run this cell first when running outside Databricks)
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("CampaignAnalysis") \
    .master("local[*]") \
    .getOrCreate()
print("Spark session ready:", spark.version)

Spark session ready: 4.1.1


In [17]:
from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler

from pyspark.ml.classification import LogisticRegression, GBTClassifier

from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

In [18]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from functools import reduce
import re

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Paths (raw files)
paths = {
    "nykaa":   "/Users/shubhvaishnav/Desktop/590/590/data/nykaa_campaign_data.csv",
    "purplle": "/Users/shubhvaishnav/Desktop/590/590/data/purplle_campaign_data.csv",
    "tira":    "/Users/shubhvaishnav/Desktop/590/590/data/tira_campaign_data.csv"
}

# Helper functions
def normalize_colname(c: str) -> str:
    c = c.strip().lower()
    c = re.sub(r"[^a-z0-9]+", "_", c)
    c = re.sub(r"_+", "_", c).strip("_")
    return c

def read_campaign(path: str, brand: str):
    df = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv(path))

    # normalize column names
    for old in df.columns:
        df = df.withColumnRenamed(old, normalize_colname(old))

    # add source column
    df = df.withColumn("brand_source", F.lit(brand))
    
    return df

# add missing columns as nulls
def align_cols(df, all_cols):
    for c in all_cols:
        if c not in df.columns:
            df = df.withColumn(c, F.lit(None))
    return df.select(*all_cols)

# Load + Union
dfs = [read_campaign(p, b) for b, p in paths.items()]
all_cols = sorted(set().union(*[set(d.columns) for d in dfs]))

dfs_aligned = [align_cols(d, all_cols) for d in dfs]
df_all = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs_aligned)

# drop exact duplicate rows
df = df_all.dropDuplicates()
df.printSchema()

root
 |-- acquisition_cost: double (nullable = true)
 |-- brand_source: string (nullable = false)
 |-- campaign_id: string (nullable = true)
 |-- campaign_type: string (nullable = true)
 |-- channel_used: string (nullable = true)
 |-- clicks: integer (nullable = true)
 |-- conversions: integer (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- date: string (nullable = true)
 |-- duration: integer (nullable = true)
 |-- engagement_score: double (nullable = true)
 |-- impressions: integer (nullable = true)
 |-- language: string (nullable = true)
 |-- leads: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- roi: double (nullable = true)
 |-- target_audience: string (nullable = true)



In [19]:
# Make date a proper date type (data is DD-MM-YYYY e.g. "24-01-2025")
df = df.withColumn("date", F.to_date(F.col("date"), "dd-MM-yyyy"))

# Create time features
df = (df
      .withColumn("year", F.year("date"))
      .withColumn("month", F.month("date"))
      .withColumn("dayofweek", F.dayofweek("date"))
)

In [20]:
# Only run if you have a single channel column like "channel_used"
channels = ["facebook", "whatsapp", "youtube", "google", "email", "instagram"]

df = df.withColumn("channel_used_lc", F.lower(F.col("channel_used")))

for ch in channels:
    df = df.withColumn(f"ch_{ch}", (F.col("channel_used_lc") == F.lit(ch)).cast("int"))

df = df.drop("channel_used_lc")

In [21]:
# Cast ROI to double
df = df.withColumn("roi_d", F.col("roi").cast("double"))

# Compute median ROI using approxQuantile
median_roi = df.approxQuantile("roi_d", [0.5], 0.001)[0]
print("Median ROI:", median_roi)

# Binary label
df = df.withColumn("label", (F.col("roi_d") > F.lit(median_roi)).cast("int"))

Median ROI: 1.23


In [22]:
categorical_cols = ["brand_source", "campaign_type", "customer_segment", "language"]

numeric_cols = ["duration", "year", "month", "dayofweek"]

channel_cols = ["ch_facebook", "ch_whatsapp", "ch_youtube", "ch_google", "ch_email", "ch_instagram"]

In [23]:
indexers = []
for c in categorical_cols:
    idx = StringIndexer()
    idx = idx.setInputCol(c).setOutputCol(c + "_idx").setHandleInvalid("keep")
    indexers.append(idx)

indexed_cols = [c + "_idx" for c in categorical_cols]

In [24]:
encoder = OneHotEncoder()
encoder = encoder.setInputCols(indexed_cols).setOutputCols([c + "_ohe" for c in categorical_cols])
ohe_cols = [c + "_ohe" for c in categorical_cols]

In [25]:
assembler_inputs = numeric_cols + channel_cols + ohe_cols

assembler = VectorAssembler()
assembler = assembler.setInputCols(assembler_inputs).setOutputCol("features").setHandleInvalid("keep")

In [26]:
# # spark.conf.set("spark.ml.temp.dir", "dbfs:/tmp/sparkml/")
# from pyspark.ml.tuning import TrainValidationSplit

# lr = LogisticRegression()
# lr = lr.setLabelCol("label").setFeaturesCol("features").setMaxIter(50)

# pipeline_lr = Pipeline(stages=indexers + [encoder, assembler, lr])

# paramGrid_lr = (ParamGridBuilder()
#     .addGrid(lr.regParam, [0.0, 0.01, 0.1])
#     .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
#     .build()
# )

# evaluator_auc = BinaryClassificationEvaluator(
#     labelCol="label",
#     rawPredictionCol="rawPrediction",
#     metricName="areaUnderROC"
# )

# # cv_lr = CrossValidator(
# #     estimator=pipeline_lr,
# #     estimatorParamMaps=paramGrid_lr,
# #     evaluator=evaluator_auc,
# #     numFolds=3,
# #     parallelism=2,
# #     collectSubModels=False
# # )

# tvs = TrainValidationSplit(
#     estimator=pipeline_lr,
#     estimatorParamMaps=paramGrid_lr,
#     evaluator=evaluator_auc,
#     trainRatio=0.8
# )

# model = tvs.fit(train_df)

# pred_lr = model.transform(test_df)

# train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# cv_lr_model = cv_lr.fit(train_df)

# pred_lr = cv_lr_model.transform(test_df)

In [27]:
import os
os.environ['SPARKML_TEMP_DFS_PATH'] = '/Users/shubhvaishnav/Desktop/590/590/mlflow_tmp'

# Sample down to 5% to fit within serverless cache limit
df_sample = df.sample(fraction=0.05, seed=42)

# Train/test split
train_df, test_df = df_sample.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count():,}  |  Test: {test_df.count():,}")

# Evaluator
evaluator_auc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

# Logistic Regression pipeline
lr = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=10,
    regParam=0.01,
    elasticNetParam=0.0
)

pipeline_lr = Pipeline(stages=indexers + [encoder, assembler, lr])
lr_model    = pipeline_lr.fit(train_df)
pred_lr     = lr_model.transform(test_df)

auc_lr = evaluator_auc.evaluate(pred_lr)
print(f"LR AUC: {auc_lr:.4f}")

Train: 6,689  |  Test: 1,640


26/03/04 21:03:40 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


LR AUC: 0.4773


In [28]:
# # spark.conf.set("spark.ml.temp.dir", "dbfs:/tmp/sparkml/")  # not supported on serverless

# lr = LogisticRegression()
# lr = lr.setLabelCol("label").setFeaturesCol("features").setMaxIter(50)

# pipeline_lr = Pipeline(stages=indexers + [encoder, assembler, lr])

# paramGrid_lr = (ParamGridBuilder()
#     .addGrid(lr.regParam, [0.0, 0.01, 0.1])
#     .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
#     .build()
# )

# evaluator_auc = BinaryClassificationEvaluator(
#     labelCol="label",
#     rawPredictionCol="rawPrediction",
#     metricName="areaUnderROC"
# )

# cv_lr = CrossValidator(
#     estimator=pipeline_lr,
#     estimatorParamMaps=paramGrid_lr,
#     evaluator=evaluator_auc,
#     numFolds=3,
#     parallelism=2,
#     collectSubModels=False
# )

# train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# # Set UC Volume path for Spark ML caching on serverless
# import os
# os.environ['SPARKML_TEMP_DFS_PATH'] = '/Volumes/workspace/default/final_project/mlflow_tmp'

# cv_lr_model = cv_lr.fit(train_df)
# pred_lr = cv_lr_model.transform(test_df)

In [29]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

def classification_report(pred_df, label_col="label", pred_col="prediction"):
    prec_eval = MulticlassClassificationEvaluator(labelCol=label_col, predictionCol=pred_col, metricName="weightedPrecision")
    rec_eval  = MulticlassClassificationEvaluator(labelCol=label_col, predictionCol=pred_col, metricName="weightedRecall")
    f1_eval   = MulticlassClassificationEvaluator(labelCol=label_col, predictionCol=pred_col, metricName="f1")

    return {
        "weightedPrecision": prec_eval.evaluate(pred_df),
        "weightedRecall": rec_eval.evaluate(pred_df),
        "f1": f1_eval.evaluate(pred_df)
    }

print("LR report:", classification_report(pred_lr))

LR report: {'weightedPrecision': 0.4871427415303901, 'weightedRecall': 0.48841463414634145, 'f1': 0.48519935536469494}


In [30]:
import pandas as pd

# Best LR model stages
best_lr_model = lr_model  # PipelineModel
best_lr = lr_model.stages[-1]   # LogisticRegressionModel

# coefficients and intercept
coef = best_lr.coefficients.toArray().tolist()
intercept = best_lr.intercept

# Build a readable feature list for numeric + channels first
base_feature_names = numeric_cols + channel_cols

# OHE creates vector columns; each vector expands into multiple positions
# We will label them as e.g. brand_source_ohe[0], brand_source_ohe[1], ...
# by using VectorAssembler metadata to get the expanded names.
assembler_model = lr_model.stages[-2]  # VectorAssembler transformer

attrs = []
try:
    # Pull expanded feature names from metadata
    meta = lr_model.transform(train_df).schema["features"].metadata
    attrs = meta["ml_attr"]["attrs"]
except Exception as e:
    attrs = None

expanded_names = []
if attrs:
    # attrs is a dict like {"numeric":[...], "binary":[...], ...}
    for k in attrs:
        for a in attrs[k]:
            expanded_names.append(a["name"])
else:
    # fallback names if metadata not available
    expanded_names = [f"feature_{i}" for i in range(len(coef))]

coef_table = pd.DataFrame({
    "feature": expanded_names,
    "beta": coef
}).sort_values("beta", ascending=False)

print("Intercept:", intercept)
display(coef_table.head(30))

Intercept: 101.70485499359458


,feature,beta
5,campaign_type_ohe_SEO,0.184961
9,customer_segment_ohe_Youth,0.184620
13,language_ohe_Tamil,0.092752
22,ch_whatsapp,0.058329
7,campaign_type_ohe_Email,0.051500
6,campaign_type_ohe_Paid Ads,0.028904
18,year,0.013772
23,ch_youtube,0.011085
25,ch_email,0.006530
21,ch_facebook,0.005017
